<a href="https://colab.research.google.com/github/TurkuNLP/intro-to-nlp/blob/master/sequence_labeling_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequence labeling (POS tagging) with MLP

This notebook builds upon the [classification with MLP notebook](https://github.com/TurkuNLP/intro-to-nlp/blob/master/mlp_imdb_hf_dset_and_trainer.ipynb) and shows how to implement a basic sequence labeling method.

---

# Setup

Install the required Python packages using [pip](https://en.wikipedia.org/wiki/Pip):

* [`transformers`](https://huggingface.co/docs/transformers/index) is a popular deep learning package primarily on top of torch
* [`datasets`](https://huggingface.co/docs/datasets/) provides support for loading, creating, and manipulating datasets
* [`evaluate`](https://huggingface.co/docs/evaluate/index) is a library of performance metrics (like accuracy etc)

In [1]:
!pip3 install -q evaluate
!pip3 install -q --upgrade transformers[torch]


In [2]:
import datasets
import evaluate
import transformers

---

# Get and prepare data

*   Let us work with the venerable, if somewhat dated [CoNLL'03 shared task](https://aclanthology.org/W03-0419.pdf) English data
*   These are English news articles, and have annotation for POS, syntactic chunks, and named entities (in the IOB format)

The data as originally distributed for the 2003 shared task has the following format:

```
Only RB B-NP O
France NNP I-NP B-LOC
and CC I-NP O
Britain NNP I-NP B-LOC
backed VBD B-VP O
Fischler NNP B-NP B-PER
's POS I-NP O
proposal NN I-NP O
. . O O
```

Here, the four space-separated columns are token text, POS tag, chunk tag, and NER tag. The goal of the original task is to predict the NER tags using the other information as features, but the dataset can be used to study predicting the other columns too.

The dataset happens to be in the HF datasets collection, so we can grab it from there


In [3]:
import torch
import transformers
import datasets

from pprint import pprint    # pretty-print

dataset = datasets.load_dataset("dany0407/conll2003")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [4]:
pprint(dataset["train"][12])

{'chunk_tags': [11, 12, 12, 12, 21, 11, 11, 12, 0],
 'id': '12',
 'ner_tags': [0, 5, 0, 5, 0, 1, 0, 0, 0],
 'pos_tags': [30, 22, 10, 22, 38, 22, 27, 21, 7],
 'tokens': ['Only',
            'France',
            'and',
            'Britain',
            'backed',
            'Fischler',
            "'s",
            'proposal',
            '.']}


As you can see above, the various labels (POS, NER and chunk tags) are converted into IDs in this dataset. We can access the textual labels of these tags through the dataset `features`:

In [5]:
POS_TAG_NAMES = dataset['train'].features['pos_tags'].feature.names
NER_TAG_NAMES = dataset['train'].features['ner_tags'].feature.names
CHUNK_TAG_NAMES = dataset['train'].features['chunk_tags'].feature.names

We can then create mappings from names to IDs and back as Python dictionaries:

In [6]:
POS2ID = { n: i for i, n in enumerate(POS_TAG_NAMES) }
ID2POS = { i: n for i, n in enumerate(POS_TAG_NAMES) }

NER2ID = { n: i for i, n in enumerate(NER_TAG_NAMES) }
ID2NER = { i: n for i, n in enumerate(NER_TAG_NAMES) }

CHUNK2ID = { n: i for i, n in enumerate(CHUNK_TAG_NAMES) }
ID2CHUNK = { i: n for i, n in enumerate(CHUNK_TAG_NAMES) }

This is what these mappings look like:

In [7]:
print(NER2ID)

{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


In [8]:
print(ID2NER)

{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}


Let's also add in explanations from Penn Treebank for the POS tags:

In [9]:
# From the documentation page and from here https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html

POS2DESCRIPTION = {
    "CC": "Coordinating conjunction",
    "CD": "Cardinal number",
    "DT": "Determiner",
    "EX": "Existential there",
    "FW": "Foreign word",
    "IN": "Preposition or subordinating conjunction",
    "JJ": "Adjective",
    "JJR": "Adjective, comparative",
    "JJS": "Adjective, superlative",
    "LS": "List item marker",
    "MD": "Modal",
    "NN": "Noun, singular or mass",
    "NNS": "Noun, plural",
    "NNP": "Proper noun, singular",
    "NNPS": "Proper noun, plural",
    "PDT": "Predeterminer",
    "POS": "Possessive ending",
    "PRP": "Personal pronoun",
    "PRP$": "Possessive pronoun",
    "RB": "Adverb",
    "RBR": "Adverb, comparative",
    "RBS": "Adverb, superlative",
    "RP": "Particle",
    "SYM": "Symbol",
    "TO": "to",
    "UH": "Interjection",
    "VB": "Verb, base form",
    "VBD": "Verb, past tense",
    "VBG": "Verb, gerund or present participle",
    "VBN": "Verb, past participle",
    "VBP": "Verb, non-3rd person singular present",
    "VBZ": "Verb, 3rd person singular present",
    "WDT": "Wh-determiner",
    "WP": "Wh-pronoun",
    "WP$": "Possessive wh-pronoun",
    "WRB": "Wh-adverb"
}

We can now try to make sense of the tags:

In [10]:
import tabulate

e = dataset["train"][12]    # work on the same example

table = []
for token, pos_id, chunk_id, ner_id in zip(e["tokens"], e["pos_tags"], e["chunk_tags"], e["ner_tags"]):
    ner_tag = ID2NER[ner_id]
    chunk_tag = ID2CHUNK[chunk_id]
    pos_tag = ID2POS[pos_id]
    pos_def = POS2DESCRIPTION.get(pos_tag,pos_tag)
    table.append([token, ner_tag, chunk_tag, pos_tag, pos_def])

print(tabulate.tabulate(table,headers=["Token", "NER", "Chunk", "POS", "POS definition"]))

Token     NER    Chunk    POS    POS definition
--------  -----  -------  -----  ------------------------
Only      O      B-NP     RB     Adverb
France    B-LOC  I-NP     NNP    Proper noun, singular
and       O      I-NP     CC     Coordinating conjunction
Britain   B-LOC  I-NP     NNP    Proper noun, singular
backed    O      B-VP     VBD    Verb, past tense
Fischler  B-PER  B-NP     NNP    Proper noun, singular
's        O      B-NP     POS    Possessive ending
proposal  O      I-NP     NN     Noun, singular or mass
.         O      O        .      .


Note that the data is organized into sentences.

---

# Create features

We'll define a simple function that takes a token sequence, the index of the focus token, and a window size and generates a few basic explicit features relevant to the task.

(Note that as we'll be predicting the POS tag, we won't look at the chunk or NER tags, which would typically only be predicted _after_ predicting POS in a "traditional" NLP pipeline)

In [11]:
def token_features(tokens, index, window_size):
    # Generate features for token in position `index` in given list of tokens
    features = []

    # Context window start and end
    window_start = max(0, index-window_size)
    window_end = min(index+window_size+1, len(tokens))    # note +1 for range

    for i in range(window_start, window_end):
          offset = i - index    # relative position
          features.append(f"token[{offset}]={tokens[i]}")

    # Example custom feature: does focus token start with an upper-case letter?
    if tokens[index][0].isupper():
        features.append("first-letter-capitalized")

    return features

We can call this function for all tokens in a sentence like so:

In [12]:
def add_features_to_sentence(sentence):
    # Collect lists of features for all tokens here
    all_features = []

    tokens = sentence["tokens"]
    for index in range(len(tokens)):
        all_features.append(" ".join(token_features(tokens, index, window_size=3)))

    return { "features": all_features}

In [13]:
for feats in add_features_to_sentence(dataset["train"][12])["features"]:
    print(feats)

token[0]=Only token[1]=France token[2]=and token[3]=Britain first-letter-capitalized
token[-1]=Only token[0]=France token[1]=and token[2]=Britain token[3]=backed first-letter-capitalized
token[-2]=Only token[-1]=France token[0]=and token[1]=Britain token[2]=backed token[3]=Fischler
token[-3]=Only token[-2]=France token[-1]=and token[0]=Britain token[1]=backed token[2]=Fischler token[3]='s first-letter-capitalized
token[-3]=France token[-2]=and token[-1]=Britain token[0]=backed token[1]=Fischler token[2]='s token[3]=proposal
token[-3]=and token[-2]=Britain token[-1]=backed token[0]=Fischler token[1]='s token[2]=proposal token[3]=. first-letter-capitalized
token[-3]=Britain token[-2]=backed token[-1]=Fischler token[0]='s token[1]=proposal token[2]=.
token[-3]=backed token[-2]=Fischler token[-1]='s token[0]=proposal token[1]=.
token[-3]=Fischler token[-2]='s token[-1]=proposal token[0]=.


The dataset is organized into sentences, so we can use the above function to add features to the entire dataset as follows.

**Note**: unlike e.g. the Python`map` function, [`Dataset.map`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasets.Dataset.map) function _updates_ its argument dataset, keeping existing values.

In [14]:
dataset = dataset.map(add_features_to_sentence)

Let's check that one more time:

In [15]:
pprint(dataset["train"][12])

{'chunk_tags': [11, 12, 12, 12, 21, 11, 11, 12, 0],
 'features': ['token[0]=Only token[1]=France token[2]=and token[3]=Britain '
              'first-letter-capitalized',
              'token[-1]=Only token[0]=France token[1]=and token[2]=Britain '
              'token[3]=backed first-letter-capitalized',
              'token[-2]=Only token[-1]=France token[0]=and token[1]=Britain '
              'token[2]=backed token[3]=Fischler',
              'token[-3]=Only token[-2]=France token[-1]=and token[0]=Britain '
              "token[1]=backed token[2]=Fischler token[3]='s "
              'first-letter-capitalized',
              'token[-3]=France token[-2]=and token[-1]=Britain '
              "token[0]=backed token[1]=Fischler token[2]='s token[3]=proposal",
              'token[-3]=and token[-2]=Britain token[-1]=backed '
              "token[0]=Fischler token[1]='s token[2]=proposal token[3]=. "
              'first-letter-capitalized',
              'token[-3]=Britain token[-2]=back

---

# Flatten dataset

The MLP code that we introduced previously expects each of the `train`, `validation` and `test` subsets of the data to consist of simple sequences of examples.

Now that we have run the feature generation, we no longer need the sentence structure and can "flatten" the data into such sequences.

In [16]:
def flatten(subset):
    # Keys for values to flatten
    keys = ["tokens", "pos_tags", "chunk_tags", "ner_tags", "features"]

    # Initialize to empty lists of tokens etc.
    flattened = { k: [] for k in keys }

    # Concatenate per-sentence lists of tokens etc.
    for sentence in subset:
        for key in keys:
            flattened[key].extend(sentence[key])

    # Return as Dataset object
    return datasets.Dataset.from_dict(flattened)

Call `flatten` for each of the subsets and make a new `DatasetDict` containing the flattened subsets:

In [17]:
flattened_dict = {
    "train": flatten(dataset["train"]),
    "validation": flatten(dataset["validation"]),
    "test": flatten(dataset["test"]),
}

flat_dataset = datasets.DatasetDict(flattened_dict)

Check that the new dataset looks OK:

In [18]:
flat_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'features'],
        num_rows: 203621
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'features'],
        num_rows: 51362
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'features'],
        num_rows: 46435
    })
})

In [19]:
for i in range(10):
    token = flat_dataset["train"]["tokens"][i]
    pos_tag = ID2POS[flat_dataset["train"]["pos_tags"][i]]
    description = POS2DESCRIPTION.get(pos_tag, pos_tag)
    features = flat_dataset["train"]["features"][i]
    print(f"{token}\t{pos_tag}\t{description}\t{features}")

EU	NNP	Proper noun, singular	token[0]=EU token[1]=rejects token[2]=German token[3]=call first-letter-capitalized
rejects	VBZ	Verb, 3rd person singular present	token[-1]=EU token[0]=rejects token[1]=German token[2]=call token[3]=to
German	JJ	Adjective	token[-2]=EU token[-1]=rejects token[0]=German token[1]=call token[2]=to token[3]=boycott first-letter-capitalized
call	NN	Noun, singular or mass	token[-3]=EU token[-2]=rejects token[-1]=German token[0]=call token[1]=to token[2]=boycott token[3]=British
to	TO	to	token[-3]=rejects token[-2]=German token[-1]=call token[0]=to token[1]=boycott token[2]=British token[3]=lamb
boycott	VB	Verb, base form	token[-3]=German token[-2]=call token[-1]=to token[0]=boycott token[1]=British token[2]=lamb token[3]=.
British	JJ	Adjective	token[-3]=call token[-2]=to token[-1]=boycott token[0]=British token[1]=lamb token[2]=. first-letter-capitalized
lamb	NN	Noun, singular or mass	token[-3]=to token[-2]=boycott token[-1]=British token[0]=lamb token[1]=.
.	.	.	

Note that this is now a single long dataset of tokens without sentence boundaries.

In [20]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import WhitespaceSplit
from transformers import PreTrainedTokenizerFast

def build_whitespace_tokenizer(dataset, vocab_size=None, min_frequency=1):
    # dataset: iterable of dicts with key "features" (string of space-delimited features)
    tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))

    # Use pure whitespace splitting, no normalization or subwording
    tokenizer.normalizer = None
    tokenizer.pre_tokenizer = WhitespaceSplit()

    special_tokens = ["[PAD]","[UNK]"]

    trainer = WordLevelTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        special_tokens=special_tokens,
    )

    iterator = (example["features"] for example in dataset)
    tokenizer.train_from_iterator(iterator, trainer=trainer)
    print(tokenizer)

    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,

        unk_token="[UNK]",
        pad_token="[PAD]",
    )

    return hf_tokenizer
tokenizer = build_whitespace_tokenizer(flat_dataset["train"], vocab_size=15000)




Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"[PAD]", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":1, "content":"[UNK]", "single_word":False, "lstrip":False, "rstrip":False, ...}], normalizer=None, pre_tokenizer=WhitespaceSplit(), post_processor=None, decoder=None, model=WordLevel(vocab={"[PAD]":0, "[UNK]":1, "first-letter-capitalized":2, "token[0]=.":3, "token[1]=.":4, ...}, unk_token="[UNK]"))


In [21]:
tokenizer.vocab

{'token[3]=you': 2172,
 'token[-2]=orders': 8137,
 'token[-1]=premium': 12989,
 'token[0]=Graf': 9346,
 'token[3]=filed': 11862,
 'token[1]=risk': 7889,
 'token[0]=verified': 7834,
 'token[3]=14': 1325,
 'token[2]=treaty': 6494,
 'token[1]=tabulated': 6771,
 'token[2]=major': 3541,
 'token[2]=religious': 11780,
 'token[-1]=rebels': 2388,
 'token[2]=1996-08-26': 4841,
 'token[-1]=200': 7201,
 'token[2]=loss': 3540,
 'token[0]=There': 2306,
 'token[-1]=only': 831,
 'token[-2]=COLORADO': 11205,
 'token[0]=Brady': 6698,
 'token[0]=there': 851,
 'token[0]=accused': 3678,
 'token[2]=joined': 11763,
 'token[2]=SCHEDULE': 9507,
 'token[1]=confirmed': 5376,
 'token[-2]=whether': 3051,
 'token[-1]=Breda': 14016,
 'token[-3]=halftime': 8202,
 'token[-1]=University': 8527,
 'token[1]=saw': 7891,
 'token[-3]=Juan': 8693,
 'token[3]=French': 2862,
 'token[-1]=ANGELES': 9058,
 'token[-3]=air': 7730,
 'token[-1]=Algeria': 9060,
 'token[2]=rule': 5412,
 'token[2]=Indian': 8391,
 'token[0]=*': 4787,
 't

In [22]:
tokenizer(flat_dataset["train"][0]["features"])

{'input_ids': [6058, 1, 1896, 5233, 2], 'attention_mask': [1, 1, 1, 1, 1]}

In [23]:
def encode(examples):
    return tokenizer(examples['features'])

dset_tokenized = flat_dataset.map(encode,batched=True,num_proc=4)

for key,val in dset_tokenized["train"][0].items():
    print(key,":",val)

Map (num_proc=4):   0%|          | 0/203621 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/51362 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/46435 [00:00<?, ? examples/s]

tokens : EU
pos_tags : 22
chunk_tags : 11
ner_tags : 3
features : token[0]=EU token[1]=rejects token[2]=German token[3]=call first-letter-capitalized
input_ids : [6058, 1, 1896, 5233, 2]
attention_mask : [1, 1, 1, 1, 1]


In [24]:
def assign_labels(ex):
    return {"labels":ex["pos_tags"]}
dset_tokenized=dset_tokenized.map(assign_labels,batched=True)

Map:   0%|          | 0/203621 [00:00<?, ? examples/s]

Map:   0%|          | 0/51362 [00:00<?, ? examples/s]

Map:   0%|          | 0/46435 [00:00<?, ? examples/s]

In [25]:
dset_tokenized["train"][0]

{'tokens': 'EU',
 'pos_tags': 22,
 'chunk_tags': 11,
 'ner_tags': 3,
 'features': 'token[0]=EU token[1]=rejects token[2]=German token[3]=call first-letter-capitalized',
 'input_ids': [6058, 1, 1896, 5233, 2],
 'attention_mask': [1, 1, 1, 1, 1],
 'labels': 22}

---

# MLP model

With the data now ready, we'll build the MLP model. Note that this is _identical_ to the MLP model we used for text classification: the only difference between the two applications is in the data.

The model class in its simplest form has `__init__()` which instantiates the layers and `forward()` which implements the actual computation. For more information on these, please see the [PyTorch turorial](https://pytorch.org/tutorials/beginner/introyt/modelsyt_tutorial.html).

In [26]:
# A Transformers library model wants a config,
# I can simply inherit from the base
# class for pretrained configs
# We don't really need to do anything very special
class MLPConfig(transformers.PretrainedConfig):
    pass

# This is the model
class MLP(transformers.PreTrainedModel):

    config_class=MLPConfig

    # In the initialization method, one instantiates the layers
    # these will be, for the most part the trained parameters of the model
    def __init__(self,config):
        super().__init__(config)
        self.all_tied_weights_keys = {} #Annoying bug in Transformers, must have this here or else we crash on saved model load
        #### HERE WE CREATE THE MODEL'S LAYERS:
        self.vocab_size=config.vocab_size #embedding matrix row count
        # Build and initialize embedding of vocab size x hidden size
        assert tokenizer.vocab["[PAD]"]==0 #let's make sure our assumption of pad==0 holds!

        self.embedding=torch.nn.Embedding(num_embeddings=self.vocab_size,embedding_dim=config.hidden_size,padding_idx=0)
        # Initialize the embeddings to random values
        # Note! This function is relatively clever and keeps the embedding for 0, the padding, pure zeros
        torch.nn.init.uniform_(self.embedding.weight.data,-0.001,0.001) #initialize the embeddings with small random values

        # This takes care of the lower half of the network, now the upper half
        # Output layer: hidden size x output size
        self.output=torch.nn.Linear(in_features=config.hidden_size,out_features=config.nlabels)
        # Now we have the parameters of the model


    # The computation of the model is put into the forward() function
    # it receives a batch of data and optionally the correct `labels`
    #
    # If given `labels` it returns (loss,output)
    # if not, then it returns (output,)
    def forward(self,input_ids,labels=None,**kwargs):
        #1) sum up the embeddings of the items
        embedded=self.embedding(input_ids) #(batch,ids)->(batch,ids,embedding_dim)
        # Since the Embedding keeps the first row of the matrix pure zeros, we don't need to worry about the padding
        # so next we sum the embeddings across the word dimension
        # (batch,ids,embedding_dim) -> (batch,embedding_dim)
        embedded_summed=torch.sum(embedded,dim=1)

        #2) apply non-linearity
        # (batch,embedding_dim) -> (batch,embedding_dim)
        projected=torch.tanh(embedded_summed) #Note how non-linearity is applied here and not when configuring the layer in __init__()

        #3) and now apply the upper, output layer of the network
        # (batch,embedding_dim) -> (batch, num_of_classes i.e. 2 in our case)
        logits=self.output(projected)

        # ...and that's all there is to it!

        #print("input_ids.shape",input_ids.shape)
        #print("embedded.shape",embedded.shape)
        #print("embedded_summed.shape",embedded_summed.shape)
        #print("projected.shape",projected.shape)
        #print("logits.shape",logits.shape)

        # If we have labels, we ought to calculate the loss
        if labels is not None:

            loss=torch.nn.CrossEntropyLoss() #This loss is meant for classification, so let's use it
            # You run it as loss(model_output,correct_labels)
            return (loss(logits,labels),logits) #2-tuple, i.e. pair of values returned
        else:
            # No labels, so just return the logits
            return (logits,) #this weird syntax means 1-tuple

Configure the model

In [27]:
num_labels = len(POS2ID)

mlp_config = MLPConfig(
    vocab_size=len(tokenizer.vocab),
    hidden_size=20,
    nlabels=num_labels
)

print("mlp_config:", mlp_config)

mlp_config: MLPConfig {
  "hidden_size": 20,
  "nlabels": 47,
  "transformers_version": "5.4.0",
  "vocab_size": 15000
}



---

# Train the model

We will use the Hugging Face [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) class for training

* Loads of arguments that control the training
* Configurable metrics to evaluate performance
* Data collator builds the batches
* Early stopping callback stops when eval loss no longer improves
* Model load/save
* Good foundation for later deep learning course
  

First, let's create a [`TrainingArguments`](https://huggingface.co/docs/transformers/v4.17.0/en/main_classes/trainer#transformers.TrainingArguments) object to specify hyperparameters and various other settings for training.

Printing this simple dataclass object will show not only the values we set, but also the defaults for all other arguments. Don't worry if you don't understand what all of these do! Many are not relevant to us here, and you can find the details in [`Trainer` documentation](https://huggingface.co/docs/transformers/main_classes/trainer) if you are interested.

In [35]:
trainer_args = transformers.TrainingArguments(
    "mlp_checkpoints", #save checkpoints here
    eval_strategy="steps",
    logging_strategy="steps",
    eval_steps=1000,
    logging_steps=1000,
    save_steps=1000,
    learning_rate=5e-4, #learning rate of the gradient descent
    max_steps=20000,
    load_best_model_at_end=True,
    per_device_train_batch_size=16
)

pprint(trainer_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1000,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=Fals

Next, let's create a metric for evaluating performance during and after training. We can use the convenience function [`load_metric`](https://huggingface.co/docs/datasets/about_metrics) to load one of many pre-made metrics and wrap this for use by the trainer.

We can use the basic `accuracy` metric, defined as the proportion of correctly predicted labels out of all labels. This time, though, the data is not evenly split.

In [36]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = np.argmax(outputs, axis=-1) #pick the index of the "winning" label
    return accuracy.compute(predictions=predictions, references=labels)

We can then create the `Trainer` and train the model by invoking the [`Trainer.train`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.train) function.

In addition to the model, the settings passed in through the `TrainingArguments` object created above (`trainer_args`), the data, and the metric defined above, we create and pass the following to the `Trainer`:

* [data collator](https://huggingface.co/docs/transformers/main_classes/data_collator): groups input into batches
* [`EarlyStoppingCallback`](https://huggingface.co/docs/transformers/main_classes/callback#transformers.EarlyStoppingCallback): stops training when performance stops improving

In [37]:
# Make a new model
mlp = MLP(mlp_config)


# Argument gives the number of steps of patience before early stopping
# i.e. training is stopped when the evaluation loss fails to improve
# certain number of times
early_stopping = transformers.EarlyStoppingCallback(5)

trainer = transformers.Trainer(
    model=mlp,
    args=trainer_args,
    train_dataset=dset_tokenized["train"],
    eval_dataset=dset_tokenized["validation"].select(range(1000)),
    compute_metrics=compute_accuracy,
    data_collator=transformers.DataCollatorWithPadding(tokenizer),
    callbacks=[early_stopping]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
1000,2.776312,2.022610,0.556000
2000,1.702751,1.352077,0.721000
3000,1.221981,1.060005,0.775000
4000,0.977140,0.893702,0.802000
5000,0.855416,0.785777,0.826000
6000,0.768623,0.706370,0.844000
7000,0.686728,0.653487,0.852000
8000,0.622537,0.617811,0.854000
9000,0.601281,0.582003,0.858000
10000,0.557751,0.558256,0.863000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20000, training_loss=0.7680243621826172, metrics={'train_runtime': 108.289, 'train_samples_per_second': 2955.056, 'train_steps_per_second': 184.691, 'total_flos': 14775739398.0, 'train_loss': 0.7680243621826172, 'epoch': 1.571462245619549})

We can then evaluate the trained model on a given dataset (here our test subset) by calling [`Trainer.evaluate`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.evaluate):

That's pretty poor performance for a task as simple as POS tagging where state-of-the-art accuracies are generally 97-99%. (The approach demonstrated in this notebook should be considered more of a teaching tool than a serious tagger implementation.)

However, the result is certainly much better than random, so we can conclude that the model is learning something about the task.

---

# What has the model learned?

* The embeddings should have some meaning to them
* Similar features should have similar embeddings

In [38]:
# Grab the embedding matrix out of the trained model
# and drop the first row (padding 0)
# then we can treat the embeddings as vectors

weights=mlp.embedding.weight.detach().cpu().numpy()

In [41]:
import sklearn
qry_idx=tokenizer.vocab["token[0]=in"]
idx2feat={v:k for k,v in tokenizer.vocab.items()}

#calculate the distance of the "in" embedding to all other embeddings
distance_to_qry=sklearn.metrics.pairwise.euclidean_distances(weights[qry_idx:qry_idx+1,:],weights)
nearest_neighbors=np.argsort(distance_to_qry) #indices of words nearest to "in"
for nearest in nearest_neighbors[0,:20]:
    print(idx2feat[nearest])

token[0]=in
token[0]=after
token[0]=at
token[0]=with
token[0]=by
token[0]=from
token[0]=on
token[0]=In
token[0]=for
token[0]=into
token[0]=between
token[0]=under
token[0]=as
token[0]=than
token[0]=since
token[0]=if
token[0]=before
token[0]=against
token[0]=through
token[0]=At


* The embeddings indeed seem to reflect the task and capture aspects of the meaning of words relevant to the task
* But now we have many classes, so we should take that into account too
* We can take the dot-product of the feature embeddings with the output layer weight of the class we care about
* When you think how the information propagates in the network, this will give us a single number reflecting each feature w.r.t. the selected label
* Technically speaking, it is the prediction of an example which only has that one feature, with respect to that one class
* Here is how we can implement it (here we rely on the fact that the model is linear, since we didn't include a nonlinearity earlier in the model's `forward()`

In [42]:
import numpy

embedding_weights=weights    #shape (features, embedding-dim)
output_weights=mlp.output.weight.detach().cpu().numpy()    #shape (num-labels, embedding-dim)

# We just matrix-multiply these together, since this gives us all the dot-products
weights_by_label=numpy.matmul(embedding_weights, output_weights.T)
weights_by_label.shape

(15000, 47)

In [43]:
def get_most_important_features_for_and_against(label):
    label_idx = POS2ID[label]
    feature_weights = weights_by_label[:,label_idx] #pick the column that interests us

    #The shape of feature_weights is (feature_vocab_size,) i.e. it is a vector
    features_weight_idx = numpy.argsort(-feature_weights) #sort in descending order, this will be vector of indices
    features_for = [idx2feat[feature_idx] for feature_idx in features_weight_idx[:20]]
    features_against = [idx2feat[feature_idx] for feature_idx in features_weight_idx[-20:][::-1]]
    return features_for, features_against

for label in ("DT", "NN", "VB"):
    dt_plus,dt_minus=get_most_important_features_for_and_against(label)
    print(f"{label}: {POS2DESCRIPTION[label]}")
    print(f"Most important features *for* label {label}:")
    pprint("   ".join(dt_plus))
    print()
    print(f"Most important features *against* label {label}:")
    pprint("   ".join(dt_minus))
    print("\n------\n")

DT: Determiner
Most important features *for* label DT:
('token[0]=the   token[0]=a   token[0]=The   token[0]=an   token[0]=this   '
 'token[0]=some   token[0]=no   token[0]=A   token[0]=any   token[0]=all   '
 'token[0]=both   token[0]=another   token[0]=those   token[0]=This   '
 'token[0]=No   token[0]=An   token[0]=these   token[0]=each   token[0]=Some   '
 'token[0]=That')

Most important features *against* label DT:
('token[0]=,   token[0]=(   token[0]=.   token[0]=of   token[0]=for   '
 'token[0]=not   token[0]=in   token[0]=from   token[0]=at   token[0]=by   '
 'token[0]=with   token[0]=/   token[0]=on   token[0]=after   token[0]=also   '
 'token[0]=as   token[0]=up   token[0]=would   token[0]=will   token[0]=out')

------

NN: Noun, singular or mass
Most important features *for* label NN:
('token[0]=percent   token[0]=government   token[0]=year   token[0]=SOCCER   '
 'token[0]=week   token[0]=state   token[0]=company   token[0]=group   '
 'token[0]=time   token[0]=spokesman   t